In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os # Import the os module

# CHANGED: added Drive mount + Colab path, since this now runs in Colab
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/f1-telemetry-ml'
labeled_path = f'{data_path}/labeled'
processed_path = f'{data_path}/processed'

target_cols = ['aggression_score', 'line_shape_score', 'oversteer_preference_score']

# CHANGED: train_data.parquet / val_data.parquet only contain label columns
# (identifiers + aggression_score/line_shape_score/oversteer_preference_score),
# NOT the flat summary features (Throttle_mean, Brake_std, etc.) this model
# needs as input. feature_cols would be empty and .fit() would fail otherwise.
# So we rebuild the flat features from the raw processed/ telemetry, merged
# against each label split, same as the LSTM notebook does for sequences.

train_labels = pd.read_parquet(f'{labeled_path}/train_data.parquet')
val_labels = pd.read_parquet(f'{labeled_path}/val_data.parquet')
test_in_dist_labels = pd.read_parquet(f'{labeled_path}/test_in_dist_data.parquet')
test_zero_shot_labels = pd.read_parquet(f'{labeled_path}/test_zero_shot.parquet')

segmented_files = glob.glob(f'{processed_path}/corners_*.parquet')
segmented = pd.concat([pd.read_parquet(f) for f in segmented_files], ignore_index=True)

SAFE_COLUMNS = ['Throttle', 'Brake', 'Speed', 'nGear', 'RPM']
merge_keys = ['year', 'race', 'session_type', 'driver', 'lap_number', 'corner_number']

def build_flat_features(labels_df, segmented_df):
    merged = segmented_df.merge(labels_df[merge_keys + target_cols], on=merge_keys, how='inner')
    rows = []
    for keys, group in merged.groupby(merge_keys):
        row = dict(zip(merge_keys, keys))
        for col in SAFE_COLUMNS:
            row[f'{col}_mean'] = group[col].mean()
            row[f'{col}_std'] = group[col].std()
            row[f'{col}_min'] = group[col].min()
            row[f'{col}_max'] = group[col].max()
        for t in target_cols:
            row[t] = group[t].iloc[0]
        rows.append(row)
    return pd.DataFrame(rows)

print("Building flat features (this takes a few minutes on the full dataset)...")
train_df = build_flat_features(train_labels, segmented)
val_df = build_flat_features(val_labels, segmented)
test_in_dist_df = build_flat_features(test_in_dist_labels, segmented)
test_zero_shot_df = build_flat_features(test_zero_shot_labels, segmented)

feature_cols = [c for c in train_df.columns if c.endswith(('_mean', '_std', '_min', '_max'))]
print(f"Train: {train_df.shape}, Val: {val_df.shape}")

# --- Train (unchanged) ---
model = MultiOutputRegressor(Ridge(alpha=1.0))
model.fit(train_df[feature_cols], train_df[target_cols])

# CHANGED: evaluate on val AND both test sets, with MAE/RMSE/R2 for each,
# instead of just val-set MAE — matches Requirement 9 (test-set metrics)
# and the baseline notebook's evaluation pattern.
def evaluate_split(df, split_name):
    preds = model.predict(df[feature_cols])
    mae = mean_absolute_error(df[target_cols], preds)
    # Fix: remove `squared=False` as it's not supported in older sklearn versions
    # and calculate RMSE by taking the square root of MSE.
    mse = mean_squared_error(df[target_cols], preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(df[target_cols], preds)
    print(f"\n--- Ridge — {split_name} ---")
    print(f"Overall MAE: {mae:.4f}  RMSE: {rmse:.4f}  R2: {r2:.4f}")
    for i, col in enumerate(target_cols):
        col_mae = mean_absolute_error(df[col], preds[:, i])
        # Fix: remove `squared=False` and calculate RMSE by taking the square root of MSE.
        col_mse = mean_squared_error(df[col], preds[:, i])
        col_rmse = np.sqrt(col_mse)
        col_r2 = r2_score(df[col], preds[:, i])
        print(f"  {col} — MAE: {col_mae:.4f}  RMSE: {col_rmse:.4f}  R2: {col_r2:.4f}")

evaluate_split(val_df, "Validation")
evaluate_split(test_in_dist_df, "Test (in-distribution)")
evaluate_split(test_zero_shot_df, "Test (zero-shot)")

# Create the directory if it doesn't exist
model_save_path = f'{data_path}/models'
os.makedirs(model_save_path, exist_ok=True)

joblib.dump(model, f'{model_save_path}/ridge_model.pkl')

Mounted at /content/drive
Building flat features (this takes a few minutes on the full dataset)...
Train: (73818, 29), Val: (32350, 29)

--- Ridge — Validation ---
Overall MAE: 0.0577  RMSE: 0.0802  R2: 0.8400
  aggression_score — MAE: 0.0527  RMSE: 0.0780  R2: 0.7359
  line_shape_score — MAE: 0.0422  RMSE: 0.0549  R2: 0.9555
  oversteer_preference_score — MAE: 0.0781  RMSE: 0.1010  R2: 0.8285

--- Ridge — Test (in-distribution) ---
Overall MAE: 0.0613  RMSE: 0.0831  R2: 0.8450
  aggression_score — MAE: 0.0634  RMSE: 0.0882  R2: 0.6925
  line_shape_score — MAE: 0.0364  RMSE: 0.0463  R2: 0.9742
  oversteer_preference_score — MAE: 0.0842  RMSE: 0.1039  R2: 0.8683

--- Ridge — Test (zero-shot) ---
Overall MAE: 0.0654  RMSE: 0.0934  R2: 0.8771
  aggression_score — MAE: 0.0740  RMSE: 0.0983  R2: 0.8018
  line_shape_score — MAE: 0.0413  RMSE: 0.0517  R2: 0.9667
  oversteer_preference_score — MAE: 0.0809  RMSE: 0.1177  R2: 0.8628


['/content/drive/MyDrive/f1-telemetry-ml/models/ridge_model.pkl']